# 04. NumPy & PyTorch 기초

## 학습 목표
- NumPy 핵심 연산(array, reshape, broadcasting, indexing) 숙달
- PyTorch Tensor의 개념과 NumPy와의 차이 이해
- Autograd(자동 미분) 메커니즘을 이해하고 활용
- Dataset/DataLoader로 데이터 파이프라인 구성
- MNIST 분류기로 전체 학습 루프를 직접 구현

## 참고 자료
- [PyTorch 공식 튜토리얼](https://pytorch.org/tutorials/)
- [NumPy 공식 문서](https://numpy.org/doc/stable/)

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. NumPy 핵심

### 1.1 Array 생성

In [ ]:
# 다양한 array 생성 방법
a = np.array([1, 2, 3])               # 리스트에서
b = np.zeros((3, 4))                   # 0으로 채운 행렬
c = np.ones((2, 3))                    # 1로 채운 행렬
d = np.eye(3)                          # 단위 행렬
e = np.arange(0, 10, 2)               # [0, 2, 4, 6, 8]
f = np.linspace(0, 1, 5)              # 0~1 균등 분할 5개
g = np.random.randn(3, 4)             # 표준정규분포
h = np.random.uniform(0, 1, (2, 3))   # 균일분포

print(f"a = {a}, shape: {a.shape}, dtype: {a.dtype}")
print(f"b shape: {b.shape}")
print(f"e = {e}")
print(f"f = {f}")
print(f"g =\n{g}")
print(f"g shape: {g.shape}, dtype: {g.dtype}")

### 1.2 Reshape

데이터의 shape을 변경. **원소의 총 개수는 유지**해야 한다.

ML에서 자주 사용:
- 이미지: (28, 28) -> (784,) (flatten)
- 배치 추가: (784,) -> (1, 784)
- CNN 입력: (784,) -> (1, 28, 28)

In [ ]:
a = np.arange(12)
print(f"원본: {a}, shape: {a.shape}")

# reshape
b = a.reshape(3, 4)
print(f"\n(3, 4):\n{b}")

c = a.reshape(2, 2, 3)
print(f"\n(2, 2, 3):\n{c}")

# -1: 자동 계산
d = a.reshape(4, -1)  # 12 / 4 = 3
print(f"\n(4, -1) -> {d.shape}:\n{d}")

# flatten: 다차원 -> 1차원
e = b.flatten()
print(f"\nflatten: {e}")

# squeeze / unsqueeze (차원 추가/제거)
f = np.array([1, 2, 3])  # shape: (3,)
g = f[np.newaxis, :]     # shape: (1, 3) - 행 벡터
h = f[:, np.newaxis]     # shape: (3, 1) - 열 벡터
print(f"\n원본 shape: {f.shape}")
print(f"newaxis (행): {g.shape} -> {g}")
print(f"newaxis (열): {h.shape} ->\n{h}")

### 1.3 Broadcasting

서로 다른 shape의 배열 간 연산을 자동으로 맞춰주는 기능.

**규칙**:
1. shape을 오른쪽 정렬
2. 각 차원이 같거나, 둘 중 하나가 1이면 broadcasting 가능
3. 1인 차원이 다른 쪽에 맞춰 확장됨

In [ ]:
# Broadcasting 예제 모음

# 1. 스칼라 + 배열
a = np.array([1, 2, 3])
print(f"[1,2,3] + 10 = {a + 10}")

# 2. (4,3) + (3,) -> 각 행에 벡터가 더해짐
X = np.ones((4, 3))
bias = np.array([10, 20, 30])
result = X + bias
print(f"\n(4,3) + (3,):")
print(f"X =\n{X}")
print(f"bias = {bias}")
print(f"X + bias =\n{result}")

# 3. (3,1) + (1,4) -> (3,4) 외적 같은 효과
row = np.array([[1], [2], [3]])  # (3, 1)
col = np.array([[10, 20, 30, 40]])  # (1, 4)
print(f"\n(3,1) + (1,4) -> {(row + col).shape}:")
print(row + col)

# 4. ML에서의 실제 사용: batch normalization 느낌
X = np.random.randn(5, 3)  # 5개 데이터, 3개 특성
mean = X.mean(axis=0)       # (3,) 각 특성의 평균
std = X.std(axis=0)         # (3,) 각 특성의 표준편차
X_normalized = (X - mean) / std  # broadcasting!
print(f"\n정규화 전 평균: {X.mean(axis=0).round(3)}")
print(f"정규화 후 평균: {X_normalized.mean(axis=0).round(10)}")
print(f"정규화 후 표준편차: {X_normalized.std(axis=0).round(3)}")

### 1.4 Indexing & Slicing

In [ ]:
a = np.arange(20).reshape(4, 5)
print(f"a =\n{a}\n")

# 기본 인덱싱
print(f"a[0, 0] = {a[0, 0]}")
print(f"a[2, 3] = {a[2, 3]}")

# 슬라이싱
print(f"\na[1:3] (행 1~2) =\n{a[1:3]}")
print(f"a[:, 2:4] (열 2~3) =\n{a[:, 2:4]}")
print(f"a[::2] (짝수 행) =\n{a[::2]}")

# Boolean indexing (조건 필터)
mask = a > 10
print(f"\na > 10 인 원소: {a[mask]}")

# Fancy indexing (인덱스 배열)
indices = np.array([0, 2, 3])
print(f"\na[[0,2,3]] (0, 2, 3번째 행) =\n{a[indices]}")

In [ ]:
# ML에서의 인덱싱 활용 예시

# 1. One-hot encoding
labels = np.array([0, 2, 1, 3, 2])  # 5개 데이터, 4개 클래스
n_classes = 4
one_hot = np.zeros((len(labels), n_classes))
one_hot[np.arange(len(labels)), labels] = 1
print("One-hot encoding:")
print(f"labels = {labels}")
print(f"one_hot =\n{one_hot}")

# 2. Softmax 출력에서 정답 클래스의 확률 추출
probs = np.array([
    [0.7, 0.1, 0.1, 0.1],
    [0.2, 0.1, 0.6, 0.1],
    [0.1, 0.8, 0.05, 0.05],
    [0.1, 0.1, 0.1, 0.7],
    [0.15, 0.15, 0.5, 0.2],
])
correct_probs = probs[np.arange(len(labels)), labels]
print(f"\n정답 클래스 확률: {correct_probs}")
print(f"Cross-entropy loss: {-np.mean(np.log(correct_probs)):.4f}")

---
## 2. NumPy -> PyTorch Tensor 변환

PyTorch의 Tensor는 NumPy array와 매우 유사하지만 두 가지 핵심 차이가 있다:

1. **GPU 연산** 지원 (CUDA)
2. **자동 미분** 지원 (Autograd)

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.backends.mps.is_available():
    print(f"MPS (Apple Silicon) available: True")

In [ ]:
# NumPy <-> PyTorch 변환

# NumPy -> Tensor
np_array = np.array([1.0, 2.0, 3.0])
tensor_from_np = torch.from_numpy(np_array)      # 메모리 공유!
tensor_copy = torch.tensor(np_array)               # 복사

print(f"NumPy:  {np_array}, dtype: {np_array.dtype}")
print(f"Tensor (from_numpy): {tensor_from_np}, dtype: {tensor_from_np.dtype}")
print(f"Tensor (tensor):     {tensor_copy}, dtype: {tensor_copy.dtype}")

# 메모리 공유 확인
np_array[0] = 999
print(f"\nnp_array 변경 후:")
print(f"np_array: {np_array}")
print(f"from_numpy (공유): {tensor_from_np}")  # 같이 변경됨!
print(f"tensor (복사): {tensor_copy}")          # 변경 안됨

# Tensor -> NumPy
t = torch.tensor([4.0, 5.0, 6.0])
n = t.numpy()  # CPU tensor만 가능
print(f"\nTensor -> NumPy: {n}")

In [ ]:
# Tensor 생성 방법 (NumPy와 유사)
a = torch.zeros(3, 4)
b = torch.ones(2, 3)
c = torch.eye(3)
d = torch.arange(0, 10, 2)
e = torch.linspace(0, 1, 5)
f = torch.randn(3, 4)       # 표준정규분포
g = torch.rand(2, 3)        # [0, 1) 균일분포

print(f"zeros: {a.shape}")
print(f"randn:\n{f}")
print(f"arange: {d}")
print(f"linspace: {e}")

### NumPy vs PyTorch 주요 차이점

| 기능 | NumPy | PyTorch |
|------|-------|---------|
| 타입 | ndarray | Tensor |
| GPU | 불가 | `.to('cuda')` |
| 자동 미분 | 불가 | `requires_grad=True` |
| 차원 추가 | `np.newaxis`, `np.expand_dims` | `unsqueeze()` |
| 차원 제거 | `np.squeeze` | `squeeze()` |
| shape 변경 | `reshape` | `reshape`, `view` |
| 전치 | `.T` | `.T`, `permute()` |
| 제자리 연산 | 없음 | `add_()`, `mul_()` (밑줄 접미사) |

---
## 3. PyTorch Tensor 연산

### 3.1 기본 연산

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

# 사칙연산 (element-wise)
print(f"a + b = {a + b}")
print(f"a * b = {a * b}  (element-wise)")
print(f"a / b = {a / b}")

# 내적
print(f"a @ b = {a @ b}")
print(f"torch.dot(a, b) = {torch.dot(a, b)}")

# 행렬 곱
A = torch.randn(3, 4)
B = torch.randn(4, 2)
C = A @ B  # 또는 torch.matmul(A, B)
print(f"\n(3,4) @ (4,2) = {C.shape}")

# 집계 연산
x = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
print(f"\nsum: {x.sum()}, mean: {x.mean()}, max: {x.max()}, argmax: {x.argmax()}")

### 3.2 GPU 이동

In [ ]:
# Device 설정 (GPU가 있으면 GPU 사용, 없으면 CPU)
if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"Using device: {device}")

# Tensor를 device로 이동
x_cpu = torch.randn(3, 3)
x_device = x_cpu.to(device)
print(f"CPU tensor device: {x_cpu.device}")
print(f"Moved tensor device: {x_device.device}")

# 처음부터 device에 생성
y = torch.randn(3, 3, device=device)
print(f"Created on device: {y.device}")

# 연산: 같은 device에 있어야 함
z = x_device + y
print(f"Result device: {z.device}")

# 다시 CPU로 (numpy 변환 전에 필요)
z_cpu = z.cpu()
z_numpy = z_cpu.numpy()
print(f"Back to CPU and NumPy: {z_numpy.shape}")

### 3.3 dtype 변환

In [ ]:
# 주요 dtype
x_float32 = torch.tensor([1.0, 2.0])    # 기본 float32
x_float64 = x_float32.double()            # float64
x_float16 = x_float32.half()              # float16 (혼합 정밀도 학습)
x_int = torch.tensor([1, 2, 3])           # int64
x_bool = torch.tensor([True, False, True]) # bool

print(f"float32: {x_float32.dtype}")
print(f"float64: {x_float64.dtype}")
print(f"float16: {x_float16.dtype}")
print(f"int64:   {x_int.dtype}")
print(f"bool:    {x_bool.dtype}")

# 변환
x = torch.tensor([1, 2, 3])  # int
x_float = x.float()          # float32로 변환
x_long = x.long()            # int64로 변환
print(f"\nint -> float: {x_float}, dtype: {x_float.dtype}")

# .to() 로도 가능
x_custom = x.to(torch.float16)
print(f".to(float16): {x_custom}, dtype: {x_custom.dtype}")

print("\n주의: 모델 학습은 보통 float32, 입력 인덱스는 long(int64)")

---
## 4. Autograd: 자동 미분

PyTorch의 가장 강력한 기능. `requires_grad=True`로 설정한 Tensor에 대해 **자동으로 gradient를 계산**해준다.

### 4.1 기본 사용법

In [ ]:
# 간단한 예제: f(x) = x^2 + 3x + 1
# f'(x) = 2x + 3

x = torch.tensor(2.0, requires_grad=True)

# Forward pass
y = x**2 + 3*x + 1
print(f"x = {x.item()}")
print(f"y = x^2 + 3x + 1 = {y.item()}")

# Backward pass (gradient 계산)
y.backward()

# gradient 확인
print(f"dy/dx = 2x + 3 = {x.grad.item()}")
print(f"직접 계산: 2*{x.item()} + 3 = {2*x.item() + 3}")

In [ ]:
# 벡터에 대한 gradient
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# f(x) = sum(x^2) = x1^2 + x2^2 + x3^2
y = (x ** 2).sum()
y.backward()

print(f"x = {x.data}")
print(f"y = sum(x^2) = {y.item()}")
print(f"dy/dx = 2x = {x.grad}")
print(f"직접: 2 * [1,2,3] = {2 * x.data}")

In [ ]:
# 행렬 연산의 gradient (선형 회귀와 유사)
# y = Wx + b 에서 W의 gradient

W = torch.randn(2, 3, requires_grad=True)
x = torch.randn(3)
b = torch.randn(2, requires_grad=True)

# Forward
y = W @ x + b      # (2,3) @ (3,) + (2,) = (2,)
loss = y.sum()      # 스칼라로 만듦 (backward는 스칼라에 대해서만)

# Backward
loss.backward()

print(f"W shape: {W.shape}")
print(f"W.grad shape: {W.grad.shape}")
print(f"W.grad =\n{W.grad}")
print(f"b.grad = {b.grad}")
print(f"\nx (입력, requires_grad=False): grad 없음 -> {x.grad}")

### 4.2 Computational Graph와 grad 누적

In [ ]:
# 중요: grad는 누적된다! 매번 zero_grad()가 필요
w = torch.tensor(2.0, requires_grad=True)

for i in range(3):
    y = w * 3
    y.backward()
    print(f"Step {i}: w.grad = {w.grad.item()}  (누적됨!)")

print("\n--- zero_grad 적용 ---")
for i in range(3):
    if w.grad is not None:
        w.grad.zero_()  # gradient 초기화!
    y = w * 3
    y.backward()
    print(f"Step {i}: w.grad = {w.grad.item()}  (초기화 후)")

In [ ]:
# gradient 계산 없이 연산하기 (추론/평가 시)
x = torch.tensor(3.0, requires_grad=True)

# 방법 1: torch.no_grad()
with torch.no_grad():
    y = x * 2
    print(f"no_grad: y.requires_grad = {y.requires_grad}")

# 방법 2: detach()
y = x.detach() * 2
print(f"detach:  y.requires_grad = {y.requires_grad}")

print("\n추론 시에는 no_grad()로 메모리와 계산 절약!")

### 4.3 Autograd로 Gradient Descent 직접 구현

In [ ]:
# Autograd로 선형 회귀 학습
np.random.seed(42)
torch.manual_seed(42)

# 데이터 생성: y = 3x + 2 + noise
X_np = np.random.uniform(-3, 3, 50)
y_np = 3 * X_np + 2 + np.random.normal(0, 0.5, 50)

X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.float32)

# 파라미터 (학습 대상)
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

lr = 0.01
loss_history = []

for epoch in range(100):
    # Forward
    y_pred = w * X + b
    loss = ((y_pred - y) ** 2).mean()  # MSE
    loss_history.append(loss.item())

    # Backward (autograd가 gradient 계산!)
    loss.backward()

    # Update (gradient 계산 없이)
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    # Gradient 초기화
    w.grad.zero_()
    b.grad.zero_()

print(f"학습 결과: y = {w.item():.4f}x + {b.item():.4f}")
print(f"실제값:    y = 3x + 2")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(loss_history)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Training with Autograd')
ax.grid(True, alpha=0.3)
plt.show()

---
## 5. Dataset & DataLoader

PyTorch의 데이터 파이프라인 구성 요소:

- **Dataset**: 데이터 하나를 가져오는 방법 정의 (`__getitem__`, `__len__`)
- **DataLoader**: Dataset을 배치 단위로 묶어서 제공 (셔플, 병렬 로딩 등)

### 5.1 커스텀 Dataset 만들기

In [ ]:
from torch.utils.data import Dataset, DataLoader

class SimpleDataset(Dataset):
    """y = 2x + 1 + noise 데이터셋"""

    def __init__(self, n_samples=100):
        self.X = torch.randn(n_samples, 1)  # (N, 1)
        self.y = 2 * self.X + 1 + 0.1 * torch.randn(n_samples, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Dataset 생성
dataset = SimpleDataset(n_samples=200)
print(f"Dataset size: {len(dataset)}")
print(f"First sample: X={dataset[0][0].item():.4f}, y={dataset[0][1].item():.4f}")

# DataLoader 생성
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# 배치 순회
for i, (X_batch, y_batch) in enumerate(dataloader):
    print(f"Batch {i}: X shape={X_batch.shape}, y shape={y_batch.shape}")
    if i >= 2:  # 처음 3개 배치만
        print(f"...총 {len(dataloader)}개 배치")
        break

In [ ]:
# 실전 패턴: CSV 파일로부터 Dataset 만들기 (시뮬레이션)

class TabularDataset(Dataset):
    """표 형태 데이터를 위한 Dataset"""

    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

# 가상 데이터: 3개 특성, 2개 클래스
np.random.seed(42)
n = 100
features = np.random.randn(n, 3).astype(np.float32)
labels = (features[:, 0] + features[:, 1] > 0).astype(np.int64)

dataset = TabularDataset(features, labels)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

X_batch, y_batch = next(iter(loader))
print(f"Features batch: {X_batch.shape}, dtype: {X_batch.dtype}")
print(f"Labels batch: {y_batch.shape}, dtype: {y_batch.dtype}")
print(f"Labels: {y_batch}")

---
## 6. MNIST 간단 분류기

지금까지 배운 모든 것을 종합하여 MNIST 손글씨 숫자 분류기를 만든다.

전체 학습 루프: **Data Loading -> Model 정의 -> Training -> Evaluation**

### 6.1 데이터 로딩

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

# 데이터 전처리: 이미지 -> Tensor, 정규화
transform = transforms.Compose([
    transforms.ToTensor(),           # [0, 255] -> [0, 1]
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST 평균/표준편차
])

# 데이터셋 다운로드
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

print(f"Training set: {len(train_dataset)} samples")
print(f"Test set: {len(test_dataset)} samples")

# 샘플 확인
image, label = train_dataset[0]
print(f"\nImage shape: {image.shape}  (C, H, W)")
print(f"Label: {label}")
print(f"Pixel range: [{image.min():.2f}, {image.max():.2f}]")

In [ ]:
# DataLoader 생성
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

# 샘플 이미지 시각화
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    image, label = train_dataset[i]
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(f'Label: {label}')
    ax.axis('off')
plt.suptitle('MNIST Sample Images', fontsize=14)
plt.tight_layout()
plt.show()

### 6.2 모델 정의

간단한 2-layer MLP (Multi-Layer Perceptron):
- 입력: 28x28 = 784
- 은닉층: 128 뉴런, ReLU
- 출력: 10 클래스

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()           # (1, 28, 28) -> (784,)
        self.fc1 = nn.Linear(784, 128)        # 첫 번째 레이어
        self.relu = nn.ReLU()                  # 활성화 함수
        self.fc2 = nn.Linear(128, 10)          # 출력 레이어

    def forward(self, x):
        x = self.flatten(x)    # (B, 1, 28, 28) -> (B, 784)
        x = self.fc1(x)        # (B, 784) -> (B, 128)
        x = self.relu(x)       # (B, 128) -> (B, 128)
        x = self.fc2(x)        # (B, 128) -> (B, 10)
        return x               # logits (softmax 전)

# 모델 생성
model = MLP()
print(model)

# 파라미터 수 확인
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n총 파라미터: {total_params:,}")
print(f"학습 가능: {trainable_params:,}")
print(f"  fc1: 784*128 + 128 = {784*128 + 128:,}")
print(f"  fc2: 128*10 + 10 = {128*10 + 10:,}")

In [ ]:
# 모델에 데이터 통과시키기 (Forward pass 테스트)
sample_batch, sample_labels = next(iter(train_loader))
print(f"입력 shape: {sample_batch.shape}")

output = model(sample_batch)
print(f"출력 shape: {output.shape}")
print(f"출력 예시 (logits): {output[0].data}")

# Softmax로 확률 변환
probs = torch.softmax(output[0], dim=0)
print(f"확률 변환: {probs.data}")
print(f"확률 합: {probs.sum().item():.4f}")
print(f"예측 클래스: {probs.argmax().item()}, 실제: {sample_labels[0].item()}")

### 6.3 학습 (Training Loop)

In [ ]:
# Device 설정
if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Training on: {device}")

# 모델, 손실함수, 옵티마이저
model = MLP().to(device)
criterion = nn.CrossEntropyLoss()           # softmax + NLL을 합친 것
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 학습 루프
n_epochs = 5
train_losses = []
train_accs = []

for epoch in range(n_epochs):
    model.train()  # 학습 모드
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (data, target) in enumerate(train_loader):
        # 1. 데이터를 device로 이동
        data, target = data.to(device), target.to(device)

        # 2. Gradient 초기화
        optimizer.zero_grad()

        # 3. Forward pass
        output = model(data)

        # 4. Loss 계산
        loss = criterion(output, target)

        # 5. Backward pass (gradient 계산)
        loss.backward()

        # 6. 파라미터 업데이트
        optimizer.step()

        # 통계
        running_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += (pred == target).sum().item()
        total += target.size(0)

    # 에폭 통계
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total
    train_losses.append(epoch_loss)
    train_accs.append(epoch_acc)

    print(f"Epoch {epoch+1}/{n_epochs} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc*100:.2f}%")

### 6.4 평가 (Evaluation)

In [ ]:
# 테스트셋에서 평가
model.eval()  # 평가 모드 (dropout, batchnorm 등 비활성화)
test_loss = 0
correct = 0
total = 0

with torch.no_grad():  # gradient 계산 비활성화 (메모리 절약)
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        test_loss += criterion(output, target).item()
        pred = output.argmax(dim=1)
        correct += (pred == target).sum().item()
        total += target.size(0)

test_loss /= len(test_loader)
test_acc = correct / total

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc*100:.2f}% ({correct}/{total})")

In [ ]:
# 학습 과정 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(range(1, n_epochs+1), train_losses, 'b-o')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(range(1, n_epochs+1), [a*100 for a in train_accs], 'r-o')
ax.axhline(test_acc*100, color='green', linestyle='--',
           label=f'Test acc: {test_acc*100:.1f}%')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Training & Test Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 예측 결과 시각화
model.eval()
test_images, test_labels = next(iter(test_loader))
test_images_device = test_images.to(device)

with torch.no_grad():
    outputs = model(test_images_device)
    predictions = outputs.argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(test_images[i].squeeze(), cmap='gray')
    color = 'green' if predictions[i] == test_labels[i] else 'red'
    ax.set_title(f'Pred: {predictions[i].item()} (True: {test_labels[i].item()})',
                 color=color, fontsize=11)
    ax.axis('off')

plt.suptitle('Model Predictions (Green=Correct, Red=Wrong)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 틀린 예측 분석
wrong_mask = predictions != test_labels
wrong_indices = torch.where(wrong_mask)[0]

if len(wrong_indices) > 0:
    n_show = min(10, len(wrong_indices))
    fig, axes = plt.subplots(1, n_show, figsize=(2*n_show, 3))
    if n_show == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        idx = wrong_indices[i]
        ax.imshow(test_images[idx].squeeze(), cmap='gray')
        ax.set_title(f'P:{predictions[idx].item()}\nT:{test_labels[idx].item()}',
                     color='red', fontsize=10)
        ax.axis('off')

    plt.suptitle(f'Wrong Predictions ({len(wrong_indices)} / {len(test_labels)} in this batch)', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("이 배치에서 틀린 예측이 없습니다!")

### 6.5 학습 루프 요약

PyTorch 학습의 기본 패턴 (거의 모든 모델에 적용):

```python
for epoch in range(n_epochs):
    model.train()                          # 학습 모드
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)  # 1. GPU 이동
        optimizer.zero_grad()              # 2. Gradient 초기화
        output = model(data)               # 3. Forward pass
        loss = criterion(output, target)   # 4. Loss 계산
        loss.backward()                    # 5. Backward pass
        optimizer.step()                   # 6. 파라미터 업데이트

    model.eval()                           # 평가 모드
    with torch.no_grad():                  # Gradient 계산 비활성화
        for data, target in test_loader:
            output = model(data)           # 예측만 수행
```

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: 모델 개선하기

위 MNIST 모델의 정확도를 높여보세요. 다음 중 하나 이상을 시도:

1. 은닉층 추가 (3-layer MLP)
2. 은닉층 크기 변경 (256, 512 등)
3. Dropout 추가 (`nn.Dropout(0.2)`)
4. 학습률 변경
5. 에폭 수 늘리기

In [ ]:
# TODO: 개선된 모델 정의
class ImprovedMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: 더 깊고 넓은 네트워크 + Dropout 추가
        pass

    def forward(self, x):
        # TODO: forward pass 구현
        pass

# TODO: 학습 루프 실행하고 기존 모델과 정확도 비교


### 연습 2: Autograd로 다항 회귀 구현

PyTorch autograd를 사용하여 $y = ax^2 + bx + c$ 모델을 학습하세요.

In [ ]:
# 데이터 생성: y = 2x^2 - 3x + 1 + noise
torch.manual_seed(42)
X = torch.linspace(-3, 3, 100)
y = 2 * X**2 - 3 * X + 1 + 0.5 * torch.randn_like(X)

# TODO:
# 1. 학습 파라미터 a, b, c 정의 (requires_grad=True)
# 2. 학습 루프: y_pred = a*X^2 + b*X + c
# 3. MSE loss 계산, backward(), 파라미터 업데이트
# 4. 학습 결과 시각화 (데이터 + 학습된 곡선)
# 5. 추정된 a, b, c 값 출력 (실제: a=2, b=-3, c=1)


---
## 핵심 정리

| 개념 | 설명 | 핵심 코드 |
|------|------|----------|
| NumPy array | ML의 기본 데이터 구조 | `np.array`, `reshape`, `broadcasting` |
| PyTorch Tensor | GPU + Autograd 지원 | `torch.tensor`, `.to(device)` |
| Autograd | 자동 미분 | `requires_grad=True`, `.backward()` |
| Dataset | 데이터 하나 가져오기 | `__getitem__`, `__len__` |
| DataLoader | 배치 단위 제공 | `DataLoader(dataset, batch_size=32)` |
| nn.Module | 모델 정의 | `__init__` + `forward` |
| 학습 루프 | 6단계 반복 | zero_grad -> forward -> loss -> backward -> step |

**핵심 패턴**: 모든 PyTorch 학습은 `zero_grad -> forward -> loss -> backward -> step` 6단계의 반복!

**다음 단계**: 이제 Transformer와 LLM의 세계로!